# Volt-VAr-Induced Curtailment Detection

Find instances where active-power curtailment is happening *because* the inverter is absorbing
reactive power (Volt-VAr response) and has run out of apparent-power headroom.

## The physics logic

An inverter rated `S_rated` kVA can deliver any combination of active power `P` and reactive
power `Q` as long as they stay inside the **apparent-power circle**: `sqrt(P² + Q²) ≤ S_rated`.

When grid voltage is high (e.g. 240 to 258 V), AS/NZS 4777.2 requires the inverter to **absorb**
reactive power. That absorption consumes part of the apparent-power "budget", so
if the irradiance is strong enough that the inverter *wants* to push out a lot of `P`, it may be
**forced to reduce `P`** to make room for the required `Q`. That reduction is labelled **Volt-VAr-induced curtailment**, distinct from Volt-Watt curtailment (which only kicks in above 253 V).

## Conditions to find Volt-VAr-induced curtailment timestamps

| # | Condition | Reason |
|---|-----------|-----|
| 1 | Clear-sky day (GHI) | so we know the inverter *could* have produced near its max |
| 2 | Peak solar window (≈11:00 to 13:00)* | when `P` is naturally highest. Most likely to hit the limit (*might be irrelevant) |
| 3 | `240 < V < 253` | V-VAr absorbing is **active** (Australia A), but V-Watt is **not yet**. Isolates the Volt-VAr effect |
| → | `sqrt(P² + Q²) ≳ S_rated` | inverter is riding its apparent-power limit |

## Two complementary tests

| Method | Uses | Answers |
|--------|------|---------|
| **A: Symptom detection** | measured P, Q, V from `ts` | *Which sites/intervals show the inverter riding its apparent-power limit while absorbing VAr?* |
| **B: Counterfactual quantification** | `all_uncurtailedpv` (GHI model) | *How many kWh of P were actually lost to VAr?* |


###  Some notes
1. **Use `S_99`, not nameplate, for the limit**: the CICCADA data pipeline normalises by `S_99`
   (99th-percentile observed apparent power), because the *effective* inverter limit in the data
   is often below the nameplate. Using nameplate could under-count curtailment.(nameplate is also pulled)
2. **Reactive sign convention.** Everything here assumes **−Q = absorbing**. If the fleet's raw
   `energy_reactive` sign is flipped, the whole detection inverts. There's a sanity-check cell.
3. **"Clear sky" needs a definition.** GHI being *high* isn't enough. It must be *stable*
   (not punching through cloud gaps). We use `GHI / GHI_clear_sky` near 1.0 **and** low local
   variance. The `structured_data` table already has `ghi` and `ghi_cs` so we use their ratio.
4. **Peak-solar window is in local time.** `t_stamp` is UTC. Convert to AEST (UTC+10) first,
   or the 11:00 to 13:00 window lands in the wrong place.
5. **Three-phase sites.** Aggregate P and Q across **all** PV circuits before computing S, and use
   `max(voltage)` across circuits (the worst phase is what triggers the VAr response).

## Before running

### 1.  `ghi` and `ghi_cs` and how they are calculated?

Both live in **`structured_data`**, the table the GHI/curtailment pipeline is built from
(`ghi_pv_estimator_general/Write_structured_data.ipynb`). Neither is a raw satellite produc. Both are derived:

- **`ghi`**: Bureau of Meteorology / NCI satellite irradiance, natively **10-minute** resolution,
  matched to each site's nearest lat/long grid cell. It is resampled to 5-minute by **duplicating**
  each 10-min reading and shifting the copy +5 minutes (not interpolated). So odd 5-minute
  intervals carry the *same* GHI value as the preceding even one. This means the practical
  temporal resolution of `ghi` is still 10 minutes, not 5.

- **`ghi_cs`** ("clear-sky GHI"): *not* a physical clear-sky model (e.g. Ineichen). It's
  **empirical**: for each site, the pipeline finds the **clearest day per month** at that
  location (lowest summed cloud-type index, `cloud_sum < 60`, peak GHI `> 200`), then for any
  given day it finds the **nearest such clear-sky day** (by calendar distance, ≤45 days). Within
  that reference day, `ghi_cs` at each 5-minute time-of-day bin is the **60th percentile of GHI**
  over a ±3-interval rolling window (`approx_percentile(GHI, 0.6) OVER (... ROWS BETWEEN 3
  PRECEDING AND 3 FOLLOWING)`), computed within continuous data segments (a gap >30 min starts a
  new segment). 
  The same logic builds `P_kw_norm_cs`, the matching clear-sky **power** shape.

So `ghi/ghi_cs` is "how much sun today, relative to the clearest day this site saw nearby in the
calendar". Not an absolute clear-sky physics model. 

This matters for you: a "clear-sky" flag near `ghi/ghi_cs ≈ 1` only means *as clear as the best recent reference day*, which might be an imperfect proxy in persistently overcast regions.


### 2. On three phase inverters

For three-phase sites, the inverter typically
responds to **per-phase** voltage, but our 240–253 V eligibility test uses `max(V)` — the worst
phase. A three-phase site could have one phase at 248 V (V-VAr-eligible) while the other two sit
at 235 V (not eligible), yet our aggregate `P` and `Q` are summed across all three. This is a
known simplification (shared with the rest of the pipeline) — it slightly overstates eligibility
for three-phase sites. If you want a cleaner read on this specific question, set
`PHASE_FILTER = "single"` to restrict to single-phase sites only, where this ambiguity doesn't
arise.

## Setup

In [ ]:
import sys
from pathlib import Path

# Search the current directory and its parents for the repository root.
_current = Path.cwd().resolve()

REPO_ROOT = next(
    (
        path
        for path in (_current, *_current.parents)
        if (path / "bms_sa_review").is_dir()
    ),
    None,
)

if REPO_ROOT is None:
    raise RuntimeError(
        "Could not locate the repository root containing 'bms_sa_review'. "
        f"Current working directory: {_current}"
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

if str(REPO_ROOT / "bms_sa_review") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "bms_sa_review"))

if str(REPO_ROOT / "bms_sa_review" / "data_query" / "lib") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "bms_sa_review" / "data_query" / "lib"))

print("Repository root:", REPO_ROOT)
print("Package exists:", (REPO_ROOT / "bms_sa_review").is_dir())

from bms_sa_review.shared.aws_config import aq, tables, databases

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
import seaborn as sns
import pytz

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid", font_scale=0.95)

In [ ]:
from bms_sa_review.data_query.lib.analysis_contract import AnalysisConfig, manifest
from bms_sa_review.shared.ciccada_config import AS4777

import bms_sa_review.data_query.lib.voltvar_queries as vq
import bms_sa_review.data_query.lib.voltvar_metrics as vm
import bms_sa_review.data_query.lib.voltvar_plots   as vpl

from bms_sa_review.data_query.lib.voltvar_params import PARAMS, describe

In [ ]:
# Analysis configuration
# Override table names to use the _flex_included rebuilds, same as notebook 02.
from bms_sa_review.shared.ciccada_config import TABLES

FLEX_INCLUDED_TABLES = dict(TABLES)
FLEX_INCLUDED_TABLES.update({
    "structured_data":         "structured_data_v2_flex_included",
    "all_uncurtailedpv":       "all_uncurtailedpv_v2_flex_included",
    "conformance_voltvar":     "conformance_voltvar_v2_flex_included",
    "conformance_voltwatt":    "conformance_voltwatt_v2_flex_included",
    "conformance_voltwattghi": "conformance_voltwattghi_v2_flex_included",
})

config = AnalysisConfig(
    tables=FLEX_INCLUDED_TABLES,
).validate()

params = PARAMS.with_changes(
    peak_hour_start=11,
    peak_hour_end=14,          # inclusive BETWEEN 11 AND 14, matching your legacy run
)

print("Analysis configuration:")
display(manifest(config))
print()
print("Volt-VAr detection parameters:")
display(describe(params))

## 1. Input coverage

Quick check: do `structured_data` and `all_uncurtailedpv` have rows for the
requested years? If a year is absent here, the corresponding Method A/B output
will be empty.

In [ ]:
coverage = vq.fetch_input_coverage(aq, config, params)
display(coverage)

## 2. Method A: Fleet-wide symptom scan

**screening** query. 

For each site-year it counts intervals that
simultaneously satisfy all of:

> GHI clear-sky filter (if enabled) **AND** peak solar window **AND**
> voltage in the Volt-VAr-only band **AND** absorbing Q **AND** operating at
> the apparent-power limit (within the tolerance band).

The result is a per-site-year table with `symptom_count` (how many intervals
show the symptom) and `headroom_displacement_kw_sum` (a first-order proxy for
lost active power, computed as the apparent-power headroom consumed by Q).

`method_a_summary` enriches this with `headroom_displacement_kwh` and flags
`affected` sites (those with ≥1 symptom interval).


**screening** query. 

In [ ]:
method_a_raw = vq.fetch_method_a_site_year(aq, config, params)
method_a_enriched, method_a_yearly = vm.method_a_summary(
    method_a_raw, config.interval_h,
)

print(f"Method A: {method_a_enriched.site_id.nunique():,} eligible sites, "
      f"{int(method_a_enriched.symptom_count.sum()):,} symptom intervals")
display(method_a_yearly)
method_a_enriched.sort_values("symptom_count", ascending=False).head(20)

### Columns breakdown

- **`symptom_count`**: 5-minute intervals this site-year spent in the detection
  window while riding the apparent-power limit (within the 4% tolerance). Higher
  = more affected.
- **`headroom_displacement_kwh`**: first-order proxy for energy lost — the
  apparent-power headroom consumed by Q, summed over symptom intervals and
  converted to kWh. This is an upper bound (assumes the inverter would have used
  all that headroom for P). Method B refines this with the counterfactual.
- **`eligible_count`**: total intervals in the detection window for this
  site-year, regardless of whether the symptom was present.
- **`avg_symptom_voltage`**, **`avg_symptom_p_kw`**, **`avg_symptom_q_kvar`**:
  average operating point during symptom intervals.

Sites at the top of this list are candidates for closer inspection in Method B.

## 3. Method B: Counterfactual attribution

Method A estimates lost energy from apparent-power headroom alone. Method B
uses the **counterfactual** (the GHI-based uncurtailed-P prediction from
`all_uncurtailedpv`) to quantify the actual displacement:

```
P_max_given_Q = sqrt(S_limit² − Q_measured²)
attributed_kW = max(0, P_counterfactual − max(P_measured, P_max_given_Q))
```

The attribution flows through four **evidence tiers**, each strictly nested
inside the previous one:

| Tier | Condition | Meaning |
|------|-----------|---------|
| 1 | Q < 0 | Inverter is absorbing reactive power |
| 2 | + at apparent-power limit | S ≈ S_limit (within tolerance) |
| 3 | + counterfactual > measured-Q headroom | The inverter *would* have produced more P absent the Q |
| 4 | + symptom present (if `require_apparent_limit_symptom`) | Attributable displacement with the strongest evidence |

**Tier 4** is the headline: intervals with both the symptom and a positive
counterfactual displacement.

### How `all_uncurtailedpv` is calculated

This is the counterfactual engine behind the whole notebook.
It's a five-stage pipeline (from the `ghi_pv_estimator_general` folder):

1. **`Write_structured_data`** builds the per-site, per-5-min feature table:
   actual power (normalised by `S_99`, not nameplate), matched `ghi`/`ghi_cs`,
   and the clear-sky power shape `P_kw_norm_cs`.
2. **`split_days`** randomly splits each site's distinct days 80/20 into
   train/validation (day-level).
3. **`model_ghi_norm`** fits, **per site and per 5-minute time-of-day bin**,
   the regression: `P_norm/P_norm_cs = a + b × (ghi/ghi_cs)`. The intercept is
   **pinned**: `a = 1 − b`, so that on a clear-sky day (`ghi/ghi_cs = 1`) the
   model reproduces the clear-sky reference shape exactly. Training only uses
   clean, uncurtailed observations: `V ≤ 253` (no V-Watt) and `P_norm ≥ 1 OR
   S_norm < 1.001` (no V-VAr curtailment already happening).
4. **`Write_All_uncartailedPV`** applies the model fleet-wide to predict
   `P_kw_norm_est`, then **floors the prediction at the actual measured value**
   (`uncurtailed_P = max(model_prediction, actual) × S_99`); curtailment can
   only ever *reduce* output, so the counterfactual is never allowed below what
   was actually measured.
5. A **MAPE < 50% quality gate** restricts the final table to sites where the
   model's accuracy is acceptable.

**Caveat**: because normalisation uses `S_99` rather than nameplate, sites that
have been persistently curtailed historically have a *depressed* `S_99`, which
understates their true uncurtailed potential.

In [ ]:
method_b_raw = vq.fetch_method_b_site_year(aq, config, params)
method_b_enriched, method_b_yearly = vm.method_b_summary(
    method_b_raw, config.interval_h,
)

print(f"Method B: {method_b_enriched.site_id.nunique():,} eligible sites, "
      f"{int(method_b_enriched.tier4_attributed_count.sum()):,} Tier 4 intervals, "
      f"{method_b_enriched.attributed_measured_q_kwh.sum():,.1f} kWh attributed")
display(method_b_yearly)

### Evidence tier funnel

In [ ]:
tiers = vm.evidence_tier_table(method_b_raw)
display(tiers)
vpl.plot_evidence_tiers(tiers);

## 4. Fleet summary, breakdowns, and concentration

Two complementary methods quantify Volt-VAr-induced active-power curtailment.

**Method A: Apparent-limit symptom scan (screening).** For every eligible
interval (peak-solar, 240–253 V, absorbing Q) Method A measures how far reactive
absorption pushed the operating point into the apparent-power circle:
`s_limit − √(s_limit² − Q²)`. It needs **no counterfactual**, only metered
telemetry and inverter geometry. So it is *model-independent* and robust to any
flaw in the uncurtailed-generation estimate. Its weakness is that it is a **loose
upper bound**: it assumes the inverter would have used all that circle-room for
active power, whether or not the sun was actually strong enough. Method A is a tool for **screening, triage, and a worst-case envelope**.

**Method B: counterfactual attribution (quantification).** Method B compares the
clear-sky counterfactual (`uncurtailed_P`, from the GHI model) against the active
power achievable while absorbing the required reactive power. It only attributes
curtailment where the counterfactual confirms real generation was lost, so it is
the **tighter estimate**, the number to report as the actual
energy impact.

> **Limitations of Method B (given the current uncurtailed-generation model).**
> _[PLACEHOLDER — to complete:_
> _• `S_99` normalisation depresses potential for historically-curtailed sites (downward bias);_
> _• GHI is used in place of POA irradiance (tilt/azimuth metadata sparse);_
> _• the clear-sky reference is an empirical nearest-clear-day, not a physical model;_
> _• the MAPE < 50% quality gate restricts coverage;_
> _• `ghi_cs_ratio_min` threshold is tunable, not derived from the standard.]_

In [ ]:
# Fleet denominators (Method A context) — legacy-equivalent, read ts
eligible_context = vq.fetch_eligible_context(aq, config, params)
all_context      = vq.fetch_all_timestamp_context(aq, config, params)

# Dual-method fleet summary with full context
vm.print_fleet_summary(
    method_a_enriched, method_a_yearly,
    method_b_enriched, method_b_yearly,
    params,
    eligible_context=eligible_context,
    all_context=all_context,
)

In [ ]:
vpl.plot_fleet_summary(
    method_a_enriched, method_a_yearly,
    method_b_enriched, method_b_yearly,
    params,
    eligible_context=eligible_context,
    all_context=all_context,
);   # trailing ;

In [ ]:
'''# Year-by-year comparison: symptom rates (Method A) vs attribution (Method B)
vpl.plot_yearly_comparison(method_a_yearly, method_b_yearly);'''

### 4.1 Breakdowns by DNSP

Are the affected sites concentrated in particular network areas? The
`group_breakdown` function cross-tabulates the Method B site-year results
against site metadata, keeping only groups with at least 20 sites to avoid
over-interpreting thin cells.

In [ ]:
meta = aq(
    "SELECT DISTINCT site_id, state, dnsp_name FROM meta_up23c WHERE is_pv = True",
    database=config.database,
)

by_dnsp_a = vm.group_breakdown_a(method_a_enriched, meta, "dnsp_name")   # Method A proxy
by_dnsp_b = vm.group_breakdown(method_b_enriched,   meta, "dnsp_name")   # Method B attribution
display(by_dnsp_b)

vpl.plot_group_energy_compare(by_dnsp_a, by_dnsp_b, "dnsp_name");

### 4.2 Concentration / Lorenz

How concentrated is the estimated curtailment? A handful of sites usually
dominate; the Lorenz curve and the top-k shares make that explicit.

In [ ]:
# Concentration for both methods (concentration() takes a metric column)
a_ranked, a_shares = vm.concentration(method_a_enriched, metric="headroom_displacement_kwh")
b_ranked, b_shares = vm.concentration(method_b_enriched)   # default: attributed_measured_q_kwh

a_total = method_a_enriched.headroom_displacement_kwh.sum()
b_total = method_b_enriched.attributed_measured_q_kwh.sum()

vpl.plot_concentration_compare(a_ranked, a_shares, a_total,
                               b_ranked, b_shares, b_total);

for name, shares, tot, rank in [("Method A", a_shares, a_total, a_ranked),
                                ("Method B", b_shares, b_total, b_ranked)]:
    print(f"\n{name}: {len(rank):,} affected sites, {tot:,.1f} kWh total")
    for pct, share in shares.items():
        print(f"  Top {pct:>2}% of sites → {share:5.1f}% of curtailment")

**Notes on plot below:**

A site-year is one site in one year: site 123456789 in 2024 is one site-year, the same site in 2025 is another. With ~14,700 sites across 2024-2025 that should be on the order of ~29,000 site-years.

Each histogram (panels E–J) only plots the positive site-years for that metric, the ones where the value is actually greater than zero (e.g. for "curtailed energy," only site-years that had some curtailment). 

Site-years with a zero value, or where the metric doesn't apply (denominator is zero, the site had no eligible intervals that year), are excluded from the bars and instead reported in the annotation box.

So the y-axis "share of positive site-years, %" answers: of all the site-years that had a positive value for this metric, what fraction fall in this bin? Each colored histogram (per year) sums to 100% across its bins (common_norm=False, stat="percent"). It's a normalized distribution shape that tells where the affected site-years cluster (e.g. "most curtailed site-years lost between 1 and 10 kWh", as per plot G), independent of how many sites were affected. The count of positive site-years and the zero/not-applicable shares are in each panel's annotation, so the distribution shape and the prevalence are reported separately rather than conflated.

In [ ]:
# Assemble the legacy Method A frames from the evidence-tier outputs
summary_by_year, overall_summary, site_year_distribution = vm.build_method_a_context(
    method_a_enriched, eligible_context, all_context, config.interval_h,
)

# Combined Method A detail (legacy Figure 1 + Figure 2 in a single block)
vpl.plot_method_a_detail(summary_by_year, overall_summary, site_year_distribution);

## 5. Visualise a day with Volt-VAr curtailment

Three power traces for a single day: **apparent power S** (the binding constraint),
**active power P** (what was actually delivered), and **reactive power |Q|** (what's
being absorbed).

The violet band marks the 240~253 V eligibility window.
The red band marks intervals where curtailment was actually detected.

This pulls a **full day** of telemetry (not just the eligible-window subset).

In [ ]:
TARGET_SITE = 276149647
TARGET_YEAR = 2025
TARGET_DATE = "2025-01-09"
# Legacy-equivalent Method B (raw ts) for the single-site plots
varcurt = vq.fetch_method_b_legacy(aq, config, params,
                                   site_id=TARGET_SITE, year=TARGET_YEAR)
total_kwh = (varcurt["varcurt_kW"] * config.interval_h).sum()
print(f"Site {TARGET_SITE}, {TARGET_YEAR}: {len(varcurt):,} Q<0 intervals, "
      f"{total_kwh:.2f} kWh VAr-induced curtailment")

# Full-day telemetry for the day plot
day_df = vq.fetch_day_data(aq, config, params,
                           site_id=TARGET_SITE, date_str=TARGET_DATE)

vpl.plot_varcurt_day(
    day_df=day_df,
    site_id=TARGET_SITE,
    date_str=TARGET_DATE,
    ac_capacity_kw=varcurt["ac_capacity_kw"].iloc[0],
    s_limit=varcurt["s_limit"].iloc[0],
    params=params,
    restrict_to_peak_hours=False,
); # Added trailing semicolon to suppress output in notebook cell, otherwise outputs 2 figures

## 6. Visualising one site: the apparent-power circle

Each eligible interval plotted on the **P–Q plane**, with the apparent-power circle
overlaid. Points sitting **on** the circle (within the 4% tolerance) are highlighted
with a red ring — these are the intervals where the inverter is genuinely riding its
limit while absorbing VAr. Colour encodes voltage.

In [ ]:
_circle = varcurt.rename(columns={
    "P_meas_kW": "P_kW", "Q_meas_kvar": "Q_kvar",
    "V_max": "V", "ac_capacity_kw": "rating_capacity",
    "P_potential_kW": "uncurtailed_P",
})
vpl.plot_apparent_power_circle(
    intervals_df=_circle,
    site_id=TARGET_SITE,
    s_limit=varcurt["s_limit"].median(),
    params=params,
); # Added trailing semicolon to suppress output in notebook cell, otherwise outputs 2 figures

In [ ]:
tot = 0
for y in (2024, 2025):
    v = vq.fetch_method_b_legacy(aq, config, params, site_id=276149647, year=y)
    k = (v["varcurt_kW"] * config.interval_h).sum()
    print(f"{y}: {len(v):,} intervals, {k:.2f} kWh")
    tot += k
print(f"Total: {tot:.2f} kWh")

## 7. Stage 2 baseline comparison

The Stage 2 Volt-VAr conformance table (`conformance_voltvar_v2`) already
carries per-site curtailment counts and sums computed during the pipeline
build. This is a quick sanity check: do the Stage 2 aggregates align with the
notebook's Method B totals? Discrepancies indicate filter differences
(e.g. flex-export exclusion, GHI filter, peak-hour window) that must be
understood before reporting.

In [ ]:
baseline = vq.fetch_stage2_vvar_baseline(aq, config, params)
display(baseline)

## 8. Notes, caveats, and next steps

**Caveats to keep in mind**

1. **`S_99` vs nameplate** — `S_99` is the empirical limit and usually the right
   choice, but for sites that have been curtailing for most of their history,
   `S_99` is itself depressed, which *under*-counts curtailment. Toggle
   `empirical_limit_basis` to compare.
2. **GHI / `all_uncurtailedpv` coverage** — Method B and the fleet-context
   denominator can't run for years outside the counterfactual table's coverage.
3. **`ghi_cs_ratio_min` and `tolerance_fraction` are tunable, not derived from
   the standard** — run a sensitivity sweep before quoting a headline number.
   Use `params.with_changes(ghi_cs_ratio_min=0.90)` and re-run.
4. **The 240–253 V band deliberately excludes the 253–258 V overlap zone** where
   V-VAr and V-Watt act together — in the overlap you can't cleanly attribute
   curtailment to VAr alone.
5. **`ghi_cs` is an empirical nearest-clear-sky-day reference, not a physical
   clear-sky model** — it inherits whatever shading/soiling/noise was present on
   that specific reference day.
6. **Peak-hour window** is currently 11:00–
   14:00 AEST. This is when P is naturally highest.

**Suggested next steps**

- Run `sensitivity_table` with different `empirical_limit_basis` / `tolerance_fraction` /
  `ghi_cs_ratio_min` values; report the range.
- Cross-tabulate affected sites by **OEM** — some inverter brands default to
  fixed-PF and won't show this symptom at all, which is itself a finding.
- Compare the Volt-VAr-curtailment site list against the Volt-Watt-curtailment
  list: sites doing *both* are losing the most generation at high voltage.

### Sensitivity example

```python
# Sweep GHI threshold
scenarios = []
for ghi_min in [0.80, 0.85, 0.90, 0.95]:
    p = params.with_changes(ghi_cs_ratio_min=ghi_min)
    raw = vq.fetch_method_b_site_year(aq, config, p)
    _, enriched = vm.method_b_summary(raw, config.interval_h)
    scenarios.append((f"GHI ≥ {ghi_min}", enriched))

display(vm.sensitivity_table(scenarios))
```

In [ ]:
iv = vq.fetch_method_b_intervals(aq, config, params, site_id=276149647, year=2024)

import numpy as np
# Legacy-style loss: potential - P_max_given_Q, on ALL Q<0 intervals
iv["legacy_varcurt"] = np.maximum(0, iv["uncurtailed_P"] - iv["pmax_measured_q_kw"])
legacy_style = (iv.loc[iv["Q_kvar"] < 0, "legacy_varcurt"] * config.interval_h).sum()

# New-style loss: potential - max(P, P_max_given_Q), only on symptom intervals
sym = iv["apparent_limit_symptom"] & (iv["Q_kvar"] < 0) & (iv["uncurtailed_P"] > iv["pmax_measured_q_kw"])
iv["new_attr"] = np.where(sym,
    np.maximum(0, iv["uncurtailed_P"] - np.maximum(iv["P_kW"], iv["pmax_measured_q_kw"])), 0)
new_style = (iv["new_attr"] * config.interval_h).sum()

print(f"Legacy-style (all Q<0, no floor):      {legacy_style:.2f} kWh  (expect ~ legacy 2024 half of 85.5)")
print(f"New-style (symptom + floor):           {new_style:.2f} kWh  (expect ~ 18.99)")
print(f"Q<0 intervals: {(iv['Q_kvar']<0).sum():,}  |  symptom intervals: {int(sym.sum()):,}")

In [ ]:
print("New Method A fleet proxy:", method_a_enriched.headroom_displacement_kwh.sum(), "kWh")
print("  (compare to legacy 103,117 — same METHOD)")
print("New Method B fleet attribution:", method_b_enriched.attributed_measured_q_kwh.sum(), "kWh")
print("  (Method B is always far smaller than Method A — tighter definition)")